# Beyond Words — Phase 2: XLS-R-300M fine-tuning (Kaggle)

Fine-tunes `facebook/wav2vec2-xls-r-300m` end-to-end for Bangla dialect identification,
on the **same train/val/test split** as the Phase 1 Whisper-embedding baseline.

**Before running — notebook settings (right sidebar):**
1. **Accelerator**: GPU T4 x2 (or P100)
2. **Internet**: ON (needed to clone the repo and download the model)
3. **Input**: attach your private dataset `bangla-accent-voice-data`

Runtime: roughly 1.5–2.5 h for 8 epochs — fits comfortably in one session.
Outputs land in `/kaggle/working`: `xlsr_best.pt`, `metrics_xlsr.json`, `test_predictions_xlsr.csv`.


In [ ]:
!git clone --depth 1 https://github.com/towfique-elahe/beyond-words.git
!nvidia-smi -L


In [ ]:
# Locate the audio root inside the attached dataset (the folder holding Barishal/, Formal/, ...)
from pathlib import Path

AUDIO_ROOT = None
for p in Path('/kaggle/input').rglob('Barishal'):
    if p.is_dir():
        AUDIO_ROOT = str(p.parent)
        break
assert AUDIO_ROOT, 'Dataset not found — attach bangla-accent-voice-data as input'
print('AUDIO_ROOT =', AUDIO_ROOT)


In [ ]:
!python beyond-words/finetune_xlsr.py \
    --manifest beyond-words/splits/manifest_split_rel.csv \
    --audio-root {AUDIO_ROOT} \
    --out-dir /kaggle/working \
    --epochs 20 --batch 8 --accum 2


If the cell above hits CUDA OOM, rerun with `--batch 4 --accum 4` (same effective batch),
or add `--grad-ckpt` (slower, much less VRAM).


In [ ]:
import json
print(json.dumps(json.load(open('/kaggle/working/metrics_xlsr.json')), indent=2))


When it finishes: **Save Version → Save & Run All** persists the outputs;
download `xlsr_best.pt`, `metrics_xlsr.json`, and `test_predictions_xlsr.csv`
from the notebook's Output tab and drop them into the local repo for the Phase 3 ablation table.
